<a href="https://colab.research.google.com/github/gustkt/my-project/blob/main/1_Pan_sharpening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pan-sharpening**

เทคนิค Pan-sharpening เป็นเทคนิคที่ใช้กับภาพถ่ายดาวเทียม ที่มี Panchromatic band เช่น THEOS-1, LANDSAT-8, LANDSAT-9 เพื่อให้มีรายละเอียดและมีประโยชน์มากขึ้น โดย Lab นี้เราจะทดลองกับข้อมูลดาวเทียม Landsat-9

ข้อมูลถ่ายที่เป็น Multi-Spectral มักจะมีรายละเอียดภาพไม่ค่อยดีนัก เมื่อเทียบกับข้อมูล Panchromatics  ดังนั้น การปรับความคมชัดของภาพช่วยได้โดยการรวมภาพเหล่านี้เข้ากับภาพขาวดำพิเศษที่มีรายละเอียดมากขึ้น ภาพขาวดำที่เรียกว่าภาพแพนโครมาติก มีรายละเอียดในระดับที่สูงกว่า แต่ไม่มีสี เราต้องการใช้รายละเอียดจากภาพแพนโครมาติกและสีจากภาพ Landsat-9 เพื่อสร้างภาพสีที่มีรายละเอียดสูงใหม่

ในการดำเนินการใน Lab นี้ นักศึกษา จำเป็นต้องตรวจสอบให้แน่ใจว่าข้อมูลทั้ง Panchromatic และ Multispectral ของ Landsat-9 อยู่ในตำแหน่งที่สมบูรณ์แบบ เพื่อให้เข้ากันได้อย่างลงตัว

มีหลายวิธีในการรวมภาพเหล่านี้ เช่น Brovey Transform, IHS Transform และ PCA วิธีการเหล่านี้ทำให้แน่ใจว่าภาพใหม่จะคงสีจาก Landsat-9 แต่ยังได้รับรายละเอียดเพิ่มเติมจากภาพแพนโครมาติกด้วย


เมื่อเราทำ Pan-sharpening เสร็จแล้ว เราก็จะได้ภาพใหม่ที่เป็นภาพสีที่มีรายละเอียดสูงขึ้น  โดยมีสีทั้งหมดจาก Landsat-9 และความคมชัดพิเศษจากภาพแบบแพนโครมาติก รูปภาพใหม่นี้สามารถนำไปใช้ได้หลายอย่าง เช่น ศึกษาพื้นดิน ค้นหาการเปลี่ยนแปลง หรือเพียงแค่ดูรายละเอียดเพิ่มเติมเกี่ยวกับโลก ดังนั้น การปรับความคมชัดของภาพเป็นวิธีหนึ่งในการทำให้ภาพจากดาวเทียมมีประโยชน์มากขึ้นสำหรับงานสำคัญทุกประเภท

In [ ]:
# ทำการ Authenticate และ initialize Earth Engine
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='ee-gustkt45513') #อย่าลืมเปลี่ยนชื่อโปรเจคของตัวเอง

In [ ]:
# กำหนดพื้นที่สนใจ
geometry = ee.Geometry.Point([98.95799098999555, 18.84423947416328])

# เรียกภาพ L9 ตัวอย่างแล้วดึง RGB และ Pan ออกมา
image = (ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA")
         .filterDate('2022-01-01', '2022-03-30')
         .filterBounds(geometry)
         .sort('CLOUD_COVER')
         .first())

In [ ]:
# ทำการ Pan-Sharp
rgb = image.select('B4', 'B3', 'B2')
pan = image.select('B8')

# แปลงเป็น HSV, สลับในแถบ PAN และแปลงกลับเป็น RGB
huesat = rgb.rgbToHsv().select('hue', 'saturation')
upres = ee.Image.cat([huesat, pan]).hsvToRgb()

In [ ]:
# สร้างแผนที่
Map = geemap.Map(center=[18.84423947416328, 98.95799098999555], zoom=14)

# แสดง เลเยอร์ ก่อนและหลังโดยใช้พารามิเตอร์ vis เดียวกัน
Map.addLayer(rgb, {'max': 0.28}, 'Original')
Map.addLayer(pan, {'max': 0.28}, 'Pan')
Map.addLayer(upres, {'max': 0.28}, 'Pansharpened')
Map


**คำถาม**
เพื่อทดสอบความเข้าใจของนักศึกษาเกี่ยวกับเทคนิคการทำ Pan-sharpening ด้วยภาพ Landsat-9 จงตอบคำถามต่อไปนี้

**1. อะไรคือเป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกล โดยเฉพาะเมื่อใช้ภาพ Landsat-9 อธิบายว่าเหตุใดจึงมีความสำคัญในการประมวลผลภาพ**

**คำตอบ** เนื่องจาก Landsat-9 เป็นดาวเทียมที่ถ่ายภาพเป็น Multi-spectral มีขนาด Pixel Size 30 เมตร ทำให้ภาพที่ได้มีความละเอียดน้อย แต่ในขณะเดียวกันก็มี Panchromatic band ซึ่งมีขนาด Pixel Size 15 เมตร แต่มีลักษณะเป็นภาพขาว-ดำ จึงเกิดเป็นเทคนิค Pan-sharpening ที่นำรายละเอียดจากภาพ Panchromatic รวมเข้ากับสีจากภาพ Landsat-9 เพื่อให้ได้ภาพสีที่มีความละเอียดสูง ซึ่งเป็นหนึ่งในเทคนิคของการปรับความคมชัดของภาพในการสำรวจระยะไกล

หากเราใช้เพียงภาพ Multi-spectral ที่มีขนาด Pixel Size 30 เมตร รายละเอียดเชิงพื้นที่อาจไม่เพียงพอต่อการนำไปใช้วิเคราะห์เชิงพื้นที่ต่าง ๆ เช่น การจำแนกการใช้ประโยชน์ที่ดิน, การติดตามการขยายตัวของเขตเมือง หรือการสังเกตความเปลี่ยนแปลงของพื้นที่ป่าชายเลน เป็นต้น ในขณะที่หากใช้เฉพาะภาพ Panchromatic ที่มี Pixel Size ละเอียดกว่าภาพ Multi-spectral ก็จะได้เพียงภาพขาว-ดำ ยากต่อการนำไปวิเคราะห์ต่อเช่นกัน การทำ Pan-sharpening ซึ่งเป็นการปรับความคมชัดของภาพจึงมีความสำคัญในการประมวลผลภาพ


**2. อธิบายขั้นตอนสำคัญที่เกี่ยวข้องกับการปรับความคมชัดด้วยเทคนิค Pan-sharpening ตั้งแต่การรับข้อมูลไปจนถึงการสร้างภาพที่ปรับความคมชัด เทคนิค Pan-sharpening มีกระบวนการสุ่มตัวอย่างใหม่ช่วยจัดแนวภาพหลายสเปกตรัมและภาพแพนโครมาติกอย่างไร**

**คำตอบ** ขั้นตอนสำคัญของการทำ Pan-sharpening มี 4 ขั้นตอน ได้แก่
1. การรับและเตรียมข้อมูล โดยเราต้องเลือกภาพจากดาวเทียมที่มีทั้ง Multi-Spectral bands และ Panchromatic band หลังจากนั้นทำการปรับแก้ภาพทั้ง radiometric correction, atmospheric correction และ geometric correction
2. การปรับ Pixel Size ให้สอดคล้องกัน
เนื่องจากภาพ Multi-Spectral มี Pixel Size 30 เมตร แต่ภาพ Panchromatic มี Pixel Size 15 เมตร จึงต้องมีการสุ่มตัวอย่างใหม่ (resampling) เพื่อปรับ Pixel Size ให้สอดคล้องกัน
3. การปรับความคมชัดของภาพด้วยเทคนิค Pan-sharpening โดยใช้วิธีการต่าง ๆ เช่น IHS (Intensity-Hue-Saturation), Brovey Transform, PCA (Principle Component Analysis) หรือ Gram-Schmidt เป็นต้น
4. ภาพที่ปรับความคมชัดแล้ว ผลลัพธ์สุดท้ายเราจะได้ภาพ Multi-Spectral ใหม่ที่มีความละเอียดเชิงพื้นที่สูงขึ้น จาก 30 เมตรเป็น 15 เมตรใน Landsat-9

โดยในขั้นตอนที่ 2 ที่มีกระบวนการสุ่มตัวอย่างใหม่ (resampling) คือกระบวนการที่ปรับขนาด Pixel และตำแหน่ง Pixel ของภาพ Multi-Spectral ให้มีความละเอียดเชิงพื้นที่และโครงสร้างกริดเดียวกันกับภาพ Panchromatic โดยภาพ Multi-Spectral จะถูกทำให้มี Pixel Size จาก 30 เมตร ให้เป็น 15 เมตร จากนั้น Pixel ใหม่ที่เกิดขึ้นจะถูกคำนวณค่าจาก Pixel เดิม ภายหลังจากทำการ resampling แล้ว Pixel ของภาพทั้งสองจะอยู่ในตำแหน่ง และพิกัดเดียวกัน ซึ่งทำให้ลดความคลาดเคลื่อนเชิงตำแหน่ง เช่น ขอบวัตถุตรง, ไม่เกิดภาพซ้อนหรือบิดเบี้ยว หรือคงคุณภาพของภาพ pan-sharpening ไว้ เป็นต้น
